## 1. python的*args和**kwargs区别
都是python中用于接收不定长参数的语法
- *args 接收的是 **位置参数**，最终变成一个tuple
- ** kwargs 接收的是 **关键字参数**，最终变成一个dict

```
def func(*args,**args):
    print(args) # (1,2,3)
    print(kwargs) # {'name':'Tom','age':18}
func(1,2,3,name="Tom",age=18)
```

使用场景；
- 参数数量不固定

参数顺序：
普通参数--*args--默认参数--**kwargs

注意：
- args和kwargs只是变量名，特殊的是* 和** 用*x,**y也是合法的
- kwargs 的key必须是字符串

## 2. list和tuple的区别
都是python的序列类型，都可以
- 存多个元素
- 支持索引
- 支持切片
- 支持遍历
核心区别是
- list 是可变对象
- tuple是不可变对象

底层区别
list 本质是动态数组，支持append,insert,remove,sort,因为需要支持动态扩容，python会预留冗余空间
tuple 只读数组，不能增删改，创建成本更低，因此查询速度略快，占用内存更少

哈希能力
tuple可以作为dict的key,list不行

注意
tuple不可变，不意味着内部所有元素都不可以变，如t=([1,2]),t[0].append(22)是合法的，因为tuple本身的结构没变

使用场景
list适合
- 数据需要修改
- 队列
- 栈
- 业务数据集合

tuple
- 配置项
- 常量数据
- 不希望被修改的数据

## 3.深拷贝和浅拷贝的区别
python变量本质上保存的是**对象引用（内存地址）**

浅拷贝
只复制最外层对象，内部嵌套对象仍然共享引用
```
a = [[1,2],[3,4]]
b = copy.deepcopy(a)
a is b # false
a[0] is b[0] # true
```

深拷贝
递归复制所有层级对象
```
a = [[1,2],[3,4]]
b = copy.copy(a)
a is b # false
a[0] is b[0] # false
```

赋值
共同指向同一个对象
浅拷贝
外层新对象，内层共享
深拷贝
都是新的

常见浅拷贝
list.copy(),数组切片 a[:]

浅拷贝适合
- 只修改外层结构，内部数据共享

深拷贝适合
- 状态隔离
- 配置副本

所谓的拷贝，本质都是处理对象的引用关系


## 4.可变对象和不可变对象
不可变对象
内存中的值不能被修改

可变对象
可以原地修改内容

本质区别
不是能不能写代码修改，而是修改后对象地址是否变化

函数传参
python参数传递的本质：传递对象的引用
所以，传可变对象，函数内部的修改会影响外部

## 5
```
def add_item(item, arr=[]):
    arr.append(item)
    return arr
print(add_item(1)) # [1]
print(add_item(2)) # [1,2]
print(add_item(3)) # [1,2,3]
```
python的默认参数，只在函数定义的时候执行一次，不是每次调用都重建

## 6.is和==的区别
== 比较的是 **值是否相等**
is 比较的是 **是否是同一个对象**


## 7.async和await是什么
是python的异步编程机制，核心是解决IO密集型场景下的等待浪费

同步：调一个网络请求，程序卡在那干等（比如等deepseek返回）,这几秒CPU啥也不敢
异步：遇到await,当前任务挂起，让出控制权，事件去跑别的任务，等IO结果出来了再继续
async定义协程函数，await标记 这里要等一个IO，等的时候可以去干别的

async/await 是 Python 的异步机制,解决 IO 密集场景的等待浪费:遇到 await 就挂起当前任务、让事件循环去跑别的,IO 回来再切回来。我的旅行 Agent 要串行调 LLM、高德、天气等多个外部 API,全是网络等待,所以从 FastAPI 路由到 LangGraph 节点到数据库访问我都用 async,这样单 worker 就能并发处理多个用户的请求。要注意的是 async 会沿调用链传染,且不能在协程里混入同步阻塞调用,否则会卡住整个事件循环;它适合 IO 密集而非 CPU 密集。

IO=Input/Output,输入/输出，程序和外部打交道的操作，外部是指除了cpu和内存之外的东西
IO类型
- 网络IO：调deepseek,查高德景点、查和风天气
- 磁盘IO：读写checkouts.bd
- 数据库IO：aiosqlite查conversations表
IO操作很慢，慢在等不是慢在算，大打个比方就是点了份外卖（发起网络请求）,等骑手送到（等IO返回）

另一类是CPU密集型：比如从1算到1亿的和、处理图片、训练模型--这些都是cpu在计算的

IO密集型（等的多）->async有用，等的时候去干别的
CPU密集型（算的多）->async 没用，因为CPU本来就没闲着

协程
一个可以**暂停**和**恢复**的函数
普通函数一旦调用就是从头跑到尾，中途不停
协程不一样，能在某个点主动暂停，把控制权交出去，过会再从暂停的地方接着跑，await 那一行就是暂停的点 意思是这里要等一个IO，我先挂起，事件循环你去忙别的
和我们的interrupt其实是很像的

## 8.python的虚拟环境是啥
虚拟环境是一个独立的python运行时沙盒，里面有自己的python解释器和包目录，与系统环境完全隔离
Agent的框架（langgraph）依赖链路极深，版本冲突是家常便饭

Python 默认所有包装在全局环境里，同一个包只能装一个版本。项目多了以后很容易冲突——你给新项目升级了某个包，旧项目就跑不起来了。

虚拟环境解决这个问题，原理是在项目目录下创建一个独立的 Python 解释器副本和独立的 site-packages 目录，激活之后 python 和 pip 命令都指向这个隔离空间。

## 9.requirements.txt和pyproject.toml区别
requirements.txt
是最简单的依赖列表，只记录包名和版本，历史悠久、兼容性好，但是功能单一
pyproject.toml
是现代python的项目配置标准，一个文件管项目元数据、依赖、构建工具、代码风格等所有配置

## 10.pip install 时发生了什么
- 解析包名 拿到所有可用版本列表和对应下载地址
- 版本解析 pip有一个依赖解析器，会把要装的包和它的子依赖一起考虑
- 下载分发包 优先下载wheel文件，因为wheel是预编译的，直接解压就行，速度快，没有的话就下载.tar.gz
- 安装，把文件解压到当前环境的site-packages目录

wheel和source包有啥区别
wheel是预打包的二进制格式，安装时候不需要编译，解压即用，source需要在本地跑setup.py或build


## 11.如何管理环境变量
本地开发
在项目根目录放一个.env文件，需要的KEY就在这里配置，在代码里面用python-dotenv的load_dotenv()读取，或者直接用os.environ.get('API_KEY'),.env必须加进.gitignore,不能提交，给个.env.example模板就行

生产环境
不用.env,直接由平台注入系统变量--dockER的--env-file

## 12.如何设计Tool Calling
- 工具定义。每个工具需要三要素：名称,参数的JOSN Schema,描述。描述是最重要的，模型完全靠描述来判断什么时候该用这个工具，写的模糊模型就会误用或者不用。参数要用JOSN schema严格定义类型，这样模型输出的参数可以直接校验。
- 调用决策。模型收到用户输入后，结合工具列表输出两种结果之一：普通文本回复，或者一个tool_call对象（包含工具名+参数）。这个决策完全由模型自主判断，代码层不干预
- 执行回路。代码层检测到模型返回了 tool_call，就在本地真正执行这个函数，拿到结果后以 tool 角色追加到消息历史，再把整个上下文送回模型，让模型基于工具结果继续推理，直到模型输出普通文本为止。
- 工程健壮性。工具执行可能失败，需要捕获异常并把错误信息返回给模型而不是直接崩溃；工具可能超时，需要设 timeout；敏感操作（比如删数据）要加 Human-in-the-loop 确认，不能让模型直接执行。

## 13.LangChain / LangGraph 的本质是什么
- langchain 本质是抽象和集成。它做了三件事：统一了各家LLM的调用接口（换模型不换业务代码）；提供了chain的概念，把prompt->模型->输出解析 串成流水线；内置了大量现成的集成，向量库、工具、文档加载器等。核心价值是减少重复代码。

- langgraph 本质是状态机。它把agent的运行过程建模成一张有向图
  - 节点（Node）是执行单元，可以是调用LLM、调用工具、人工审批；
  - 变（Edge）是流转条件
  - State是贯穿整个图的共享数据结构。
每次运行就是从入口节点开始，按条件再图上游走，直到到达终止节点。
它解决的核心问题是：普通的ReAct Agent是个while循环，模型每步自己决定下一步，流程不透明、难以干预。LangGraph把流程显式化---能看到图的结构，能在任意节点加断点，能持久化中间状态做断点续跑。

State是一个TypedDict,贯穿整图的所有节点，每个节点都可以读取和更新它
Langchain 负责 “调用什么能力”，Langgraph负责这些能力按什么流程、带什么状态运行

## 14.Agent 为什么会死循环
Agent的运行模式是LLM推理-> 调工具-> 结果喂回 -> 继续推理的循环，终止的条件完全依赖模型判断 模型完成了。如果模型判断失误、工具一直返回错误、或者目标本身有歧义，模型就会持续调用工具找不到停下来的理由

死循环有几种典型原因
- 工具返回结果无效但模型不放弃。比如搜索工具没找到结果，返回‘未找到’，模型认为任务没有完成，换个关键词继续搜，反复失败反复重试。
- 目标描述有歧义。用户说‘帮我找到最好的酒店’，模型不知道最好的标准，每次觉得还能再优化，不断的调用工具比较，永远不满足条件。
- 工具之间形成依赖环。比如工具A触发工具B，工具B触发工具A。
- 模型幻觉。模型错误的认为上一步工具调用失败，重复调用一个工具，实际已经成功。

如何防止？
- 最基本的是设置最大步数上限，recursion_limit，超过直接抛异中止
- 工具层面做幂等和明确返回，工具执行成功就在返回文本里面明确写‘已完成‘，执行失败就说清楚原因和建议，不给模型重试的空间。
- 在prompt里约束终止条件，明确告诉模型什么情况下应该停止，比如‘如果工具连续返回空结果超过两次，直接给出现有信息的回答’。
- langgraph 的图结构本身是一道防线，因为流程是显式的，不会出现工具之间形成依赖环，模型只能沿着画的边走。

死循环不止会出现在工具调用阶段，推理、多agent调用、反思节点都可能出现，本质原因都一样，缺少明确的终止条件，所以recursion_limit所有阶段的统一兜底。

## 15. 如何做 Agent Memory
Agent Memory 分四种：对话历史（短期记忆）、外部存储（长期记忆）、实体记忆（结构化存储用户信息）、工作记忆（当次任务的中间状态）。
核心设计问题：什么信息需要跨会话保留、用什么存储、上下文窗口满了怎么压缩。

- 短期记忆（对话历史）。就是messages列表，当前会话内模型可以看到完整的来回对话。问题是上下文窗口有限，对话长了会超出token限制，解决方案是做summary Memory--把早期对话压缩成摘要，只保留最近N条完整消息。
- 长期记忆（跨会话持久化）。用户下次打开应用，Agent还记得上次的偏好和结论。实现上，把关键信息存数据库，每次会话开始检索相关记忆注入Prompt.Langgraph提供了MemorySaver做checkpoint,可以持久化整个图的状态。
- 实体记忆。专门存结构化的用户画像，比如用户喜欢历史文化景点，预算中等，以KV形式维护，每次对话更新，下次召回。
- 工作记忆。就是当次任务的中间状态，Langgraph里对应State--项目里存的景点列表、天气、酒店，节点之间共享，任务结束就丢弃。

上下文窗口满了怎么办？
- 截断，简单粗暴
- summary 把早期对话压缩成一段摘要
- RAG召回 把历史存向量库，每次只检索相关片段注入

